In [1]:
# WC 2026 – xG Visualizations by Country & Player
### Data: Understat Big-5 Leagues | Seasons 21/22 – 25/26

In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv("wc2026_outputs/understat_big5_player_season_2122_2526_full_merged_intl.csv")

# Aggregate per player (sum xG across all seasons they appear)
player_df = (
    df.groupby(["player", "nation"], as_index=False)
    .agg(
        total_xg=("xg", "sum"),
        total_np_xg=("np_xg", "sum"),
        total_goals=("goals", "sum"),
        total_assists=("assists", "sum"),
        total_shots=("shots", "sum"),
        total_minutes=("minutes", "sum"),
        seasons=("season", "nunique"),
    )
    .sort_values("total_xg", ascending=False)
)

# Nation-level aggregates
nation_df = (
    player_df.groupby("nation", as_index=False)
    .agg(
        nation_total_xg=("total_xg", "sum"),
        num_players=("player", "count"),
        top_player_xg=("total_xg", "max"),
    )
    .sort_values("nation_total_xg", ascending=False)
)

print(f"Players: {len(player_df):,}  |  Nations: {player_df['nation'].nunique()}")
player_df.head(10)


Players: 3,216  |  Nations: 104


,player,nation,total_xg,total_np_xg,total_goals,total_assists,total_shots,total_minutes,seasons
843,Erling Haaland,Norway,128.231064,106.177592,129,30,532,12399,5
1119,Harry Kane,England,126.231913,100.433580,139,34,622,13865,5
1681,Kylian Mbappe-Lottin,France,125.869327,100.328809,139,36,689,12998,5
2581,Robert Lewandowski,Poland,124.436990,111.728776,115,21,552,12340,5
2162,Mohamed Salah,Egypt,104.432408,83.880922,94,59,561,13818,5
1697,Lautaro Martínez,Argentina,93.851186,85.476892,92,19,538,12121,5
1434,Jonathan Christian David,Canada,90.167628,71.164123,79,17,362,12638,5
2773,Serhou Guirassy,France,81.669907,71.818810,81,6,337,9318,5
2353,Ollie Watkins,England,78.364979,74.559130,69,30,409,14240,5
142,Alexander Sørloth,Norway,67.424406,66.681128,69,11,305,9649,5


## Chart 1 – Total xG per Nation (Top 30)
Bar chart showing cumulative xG generated by all players from each nation across all Big-5 league seasons.

In [3]:
top30_nations = nation_df.head(30).sort_values("nation_total_xg")

fig1 = go.Figure(
    go.Bar(
        x=top30_nations["nation_total_xg"],
        y=top30_nations["nation"],
        orientation="h",
        marker=dict(
            color=top30_nations["nation_total_xg"],
            colorscale="Plasma",
            showscale=True,
            colorbar=dict(title="Total xG"),
        ),
        text=top30_nations["nation_total_xg"].round(1),
        textposition="outside",
        customdata=top30_nations[["num_players", "top_player_xg"]].values,
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Total xG: %{x:.1f}<br>"
            "Players in dataset: %{customdata[0]}<br>"
            "Best single player xG: %{customdata[1]:.1f}<extra></extra>"
        ),
    )
)

fig1.update_layout(
    title=dict(text="Total xG by Nation – Big-5 Leagues (2021/22 – 2025/26)", font_size=18),
    xaxis_title="Cumulative xG (all seasons)",
    yaxis_title="",
    height=750,
    template="plotly_dark",
    margin=dict(l=180, r=60, t=70, b=50),
)

fig1.show()


## Chart 2 – Player xG Bubble Chart (Top 25 Nations)
Each bubble is one player. Position on X-axis = nation (sorted by total nation xG). Bubble size and Y-axis = player's total xG. Hover for full details.

In [4]:
top25_nation_list = nation_df.head(25)["nation"].tolist()
bubble_df = player_df[player_df["nation"].isin(top25_nation_list)].copy()

# Nation rank order for x-axis
nation_order = nation_df[nation_df["nation"].isin(top25_nation_list)]["nation"].tolist()

fig2 = px.strip(
    bubble_df,
    x="nation",
    y="total_xg",
    color="nation",
    hover_name="player",
    hover_data={
        "total_xg": ":.1f",
        "total_goals": True,
        "total_assists": True,
        "total_shots": True,
        "total_minutes": True,
        "seasons": True,
        "nation": False,
    },
    labels={
        "total_xg": "Total xG",
        "nation": "Nation",
        "total_goals": "Goals",
        "total_assists": "Assists",
        "total_shots": "Shots",
        "total_minutes": "Minutes",
        "seasons": "Seasons",
    },
    category_orders={"nation": nation_order},
    title="Player xG Distribution by Nation – Top 25 Nations (2021/22 – 2025/26)",
    height=650,
    template="plotly_dark",
)

fig2.update_traces(marker=dict(size=5, opacity=0.75))
fig2.update_layout(
    showlegend=False,
    xaxis=dict(tickangle=-45),
    yaxis_title="Total xG (all seasons combined)",
    margin=dict(b=120),
)

fig2.show()


## Chart 3 – Top 10 Players per Nation (Faceted Bar Chart)
Top 10 xG earners for each of the Top 20 nations. Each facet = one country.

In [5]:
top20_nations = nation_df.head(20)["nation"].tolist()

top10_per_nation = (
    player_df[player_df["nation"].isin(top20_nations)]
    .groupby("nation", group_keys=False)
    .apply(lambda g: g.nlargest(10, "total_xg"))
    .reset_index(drop=True)
)

fig3 = px.bar(
    top10_per_nation,
    x="total_xg",
    y="player",
    color="total_xg",
    facet_col="nation",
    facet_col_wrap=4,
    orientation="h",
    color_continuous_scale="Viridis",
    labels={"total_xg": "Total xG", "player": ""},
    hover_data={
        "total_goals": True,
        "total_assists": True,
        "total_shots": True,
        "seasons": True,
        "nation": False,
    },
    title="Top 10 Players by xG – Per Nation (Top 20 Nations | 2021/22 – 2025/26)",
    height=1800,
    template="plotly_dark",
)

fig3.update_layout(
    coloraxis_showscale=False,
    showlegend=False,
    margin=dict(l=130, t=80, b=40),
)
fig3.update_yaxes(matches=None, showticklabels=True, automargin=True)
fig3.update_xaxes(matches=None, showticklabels=True)
fig3.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig3.show()


/var/folders/2s/tlbj4v1n6dxbg5403wzxdv_40000gn/T/ipykernel_33847/1647236736.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.nlargest(10, "total_xg"))


## Chart 4 – xG vs Goals Scatter (All Players, Top 25 Nations)
Each point is a player. Color = nation. Diagonal dashed line = perfect xG conversion. Points **above** the line = overperformers; **below** = underperformers.

In [6]:
scatter_df = player_df[
    (player_df["nation"].isin(top25_nation_list)) & (player_df["total_xg"] > 1)
].copy()
scatter_df["xg_overperformance"] = scatter_df["total_goals"] - scatter_df["total_xg"]

fig4 = px.scatter(
    scatter_df,
    x="total_xg",
    y="total_goals",
    color="nation",
    size="total_shots",
    size_max=22,
    hover_name="player",
    hover_data={
        "total_xg": ":.1f",
        "total_goals": True,
        "total_assists": True,
        "xg_overperformance": ":.1f",
        "seasons": True,
        "nation": True,
        "total_shots": True,
    },
    labels={
        "total_xg": "Total xG",
        "total_goals": "Total Goals",
        "xg_overperformance": "Goals – xG",
        "total_shots": "Total Shots",
        "seasons": "Seasons",
    },
    category_orders={"nation": nation_order},
    title="xG vs Goals Scored per Player – Top 25 Nations (bubble size = total shots)",
    height=700,
    template="plotly_dark",
    opacity=0.8,
)

# Perfect conversion diagonal
max_val = scatter_df[["total_xg", "total_goals"]].max().max() + 5
fig4.add_shape(
    type="line",
    x0=0, y0=0, x1=max_val, y1=max_val,
    line=dict(color="white", dash="dash", width=1.2),
)
fig4.add_annotation(
    x=max_val * 0.9, y=max_val * 0.85,
    text="Perfect xG conversion",
    showarrow=False,
    font=dict(color="lightgrey", size=11),
)

fig4.update_layout(
    legend=dict(title="Nation", orientation="v", x=1.01),
    margin=dict(r=180),
)

fig4.show()
